all function taken from utils_v5_extra_data.py and test_5_extra_data.py

In [7]:
import json
from pathlib import Path
from datetime import timedelta
import time
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.lines import Line2D
import tempfile
import shutil
from tqdm import tqdm

import rasterio
import rioxarray
from rasterio.transform import rowcol
from rioxarray.exceptions import NoDataInBounds
import geopandas as gpd

from pyproj import CRS, Transformer, Geod

from PIL import Image as PILImage

# CyFi imports
from cyfi.pipeline import CyFiPipeline
from cyfi.config import FeaturesConfig
from cyfi.data.features import generate_all_features
from cyfi.cli import DEFAULT_MODEL_PATH

# images in excel
import io

import planetary_computer as pc
from pystac_client import Client


import pandas as pd
import os
import sys
from pathlib import Path

In [43]:
import os
import numpy as np
import rasterio
import torch
import torch.nn as nn
from torchvision import models
import pytorch_lightning as L
import timm

In [2]:


def get_bounding_box(latitude, longitude, meter_buffer=50000):
    """
    Given a latitude, longitude, and buffer in meters, returns a bounding
    box [min_lon, min_lat, max_lon, max_lat] around the point.
    """
    g = Geod(ellps='WGS84')
    # Forward calculation: (lon, lat, back_az)
    lon_west, _, _ = g.fwd(longitude, latitude, 270, meter_buffer)
    _, lat_south, _ = g.fwd(longitude, latitude, 180, meter_buffer)
    lon_east, _, _ = g.fwd(longitude, latitude, 90, meter_buffer)
    _, lat_north, _ = g.fwd(longitude, latitude, 0, meter_buffer)
    return [lon_west, lat_south, lon_east, lat_north]

def get_date_range(date, time_buffer_days=15):
    """Get a date range to search for in the planetary computer based
    on a sample's date. The time range will include the sample date
    and time_buffer_days days prior

    Returns a string"""
    datetime_format = "%Y-%m-%d"
    range_start = pd.to_datetime(date) - timedelta(days=time_buffer_days)
    range_end = pd.to_datetime(date) + timedelta(days=time_buffer_days)
    date_range = f"{range_start.strftime(datetime_format)}/{range_end.strftime(datetime_format)}"

    return date_range

def search_with_retry(search_obj, max_retries=5):
    """
    Executes search.item_collection() with retries to handle API timeouts.
    """
    for attempt in range(max_retries):
        try:
            # item_collection() is the modern replacement for get_all_items()
            return search_obj.item_collection()
        except Exception as e:
            error_msg = str(e)
            # Check for timeout or server availability errors
            if "maximum allowed time" in error_msg or "504" in error_msg or "503" in error_msg:
                wait_time = (2 ** attempt) + (random.random() * 2)
                print(f"   >>> API Timeout/Error. Retrying in {wait_time:.2f}s (Attempt {attempt + 1}/{max_retries})...")
                time.sleep(wait_time)
            else:
                # If it's a logic error (not network/timeout), raise immediately
                raise e
                
    print(f"   >>> Failed after {max_retries} retries. Skipping this interval.")
    return []

def find_satelite_images(p_lat, p_lon, sel_date, meter_buffer = 3000):

    catalog = Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1", modifier=pc.sign_inplace
    )
    
    bbox = get_bounding_box(p_lat, p_lon, meter_buffer)
    print(f"Search BBox: {bbox}")
    date_range = get_date_range(sel_date)

    search = catalog.search(
        collections=["sentinel-2-l2a"], bbox=bbox, datetime=date_range
    )

    items = search_with_retry(search)
    
    item_details = pd.DataFrame(
        [
            {
                "datetime": item.datetime.strftime("%Y-%m-%d"),
                "platform": item.properties["platform"],
                "min_long": item.bbox[0],
                "max_long": item.bbox[2],
                "min_lat": item.bbox[1],
                "max_lat": item.bbox[3],
                "bbox": item.bbox,
                "item_obj": item,
            }
            for item in items
        ]
    )

    # check which rows actually contain the sample location
    item_details["contains_sample_point"] = (
        (item_details.min_lat < p_lat)
        & (item_details.max_lat > p_lat)
        & (item_details.min_long < p_lon)
        & (item_details.max_long > p_lon)
    )

    print(
        f"Filtering from {len(item_details)} returned to {item_details.contains_sample_point.sum()} items that contain the sample location"
    )
    
    item_details = item_details[item_details["contains_sample_point"]]
    item_details[["datetime", "platform", "contains_sample_point", "bbox"]].sort_values(
        by="datetime"
    )

    item_details["per_clouds"] = -1.0
    # the commented code below finds the cloud coverage of the whole image
    '''
    for it, row in item_details.iterrows():
        ar = rioxarray.open_rasterio(pc.sign(row.item_obj.assets["SCL"].href)).to_numpy()
        mask = (ar == 3) | ((ar >= 7) & (ar <= 10))
        cloud_per = round(100 * np.sum(mask) / ar.size, 2)
        item_details.at[it, "per_clouds"] = cloud_per
    '''
    
    # the code below finds the cloud coverage of the box of interest
    for it, row in item_details.iterrows():
        try:
            # 1. Lazy load the SCL asset URL
            scl_href = pc.sign(row.item_obj.assets["SCL"].href)
            
            # 2. Open and Clip to the bounding box (Lazy loading)
            # This ensures we don't download the whole tile, just the area of interest
            ds = rioxarray.open_rasterio(scl_href)
            ds_clip = ds.rio.clip_box(
                minx=bbox[0], miny=bbox[1], maxx=bbox[2], maxy=bbox[3], crs="EPSG:4326"
            )
            
            # 3. Load values into memory (now it's a small 2D array)
            ar = ds_clip.values.squeeze()
            
            # 4. Calculate Cloud Percentage
            # We filter for: 
            #   3: Cloud Shadows
            #   7: Unclassified
            #   8: Cloud Medium Probability
            #   9: Cloud High Probability
            #   10: Cirrus
            # Create a boolean mask
            cloud_mask = (ar == 3) | ((ar >= 7) & (ar <= 10))
            
            # Calculate percentage: (Count of True / Total Pixels) * 100
            cloud_per = round(100 * np.sum(cloud_mask) / ar.size, 2)
            
            item_details.at[it, "per_clouds"] = cloud_per            
        except Exception as e:
            print(f"Error calculating clouds for item {it}: {e}")
    
    item_details['date_difference'] = (pd.to_datetime(item_details['datetime']) - pd.to_datetime(sel_date)).dt.days
    
    return item_details

def select_item(item_details):
    
    if item_details.empty:
        return False, None
    
    low_clouds = item_details[item_details.per_clouds < 7.5]
    if len(low_clouds) == 0:
        return False, None
    
    # Sort by lowest clouds
    sel_items = low_clouds.sort_values(by="per_clouds", ascending=True)
    best_row = sel_items.iloc[0]
    
    return True, best_row["item_obj"]

def extract_and_save_tile(
    item,
    lat,
    lon,
    case_id,
    initial_pixel_size=100,
    save_data=False,
    output_path=None, 
    print_images=True):
    
    print("--- Starting Tile Extraction and Processing (Single Center) ---")
    
    # 1. Setup CRS and Center
    target_crs = CRS.from_string(item.properties["proj:code"])
    center_lat = lat
    center_lon = lon
    print(f"Center Point (Lat/Lon): ({center_lat:.4f}, {center_lon:.4f})")
    
    # 2. Calculate Clipping Box in UTM 
    # We create the transformer early to convert the single center point
    transformer_latlon_to_utm = Transformer.from_crs(CRS.from_string("EPSG:4326"), target_crs, always_xy=True)
    center_x, center_y = transformer_latlon_to_utm.transform(center_lon, center_lat)

    # Calculate box based on initial_pixel_size * 10m
    final_pixel_side = int(initial_pixel_size)
    initial_side_meters = final_pixel_side * 10
    half_side = initial_side_meters / 2
    
    min_x_final = center_x - half_side
    max_x_final = center_x + half_side
    min_y_final = center_y - half_side
    max_y_final = center_y + half_side
    
    print(f"Final Tile Dimension (10m pixels): {final_pixel_side} x {final_pixel_side}")
    
    clip_bbox_utm = (min_x_final, min_y_final, max_x_final, max_y_final)

    # --- 3. PREPARE METADATA ---
    # Simplified to just store the current case info since points_df is gone
    points_data_list = [{
        'case': str(case_id.split("_")[0]),
        'lat': center_lat,
        'lon': center_lon,
        'date': str(case_id.split("_")[1])
    }]

    # 5. Asset Clipping
    band_gsd_map = {
        "B02": 10, "B03": 10, "B04": 10, "B08": 10, "visual": 10, "AOT": 10, "WVP": 60,
        "B05": 20, "B06": 20, "B07": 20, "B8A": 20, "SCL": 20, "B11": 20, "B12": 20,
        "B01": 60, "B09": 60
    }
    
    clipped_data = {}
    print("--- Clipping and Aligning Rasters ---")
    
    try:
        b04_href = pc.sign(item.assets["B04"].href)
        b04_ds = rioxarray.open_rasterio(b04_href)
        b04_clip = b04_ds.rio.clip_box(
            minx=clip_bbox_utm[0], miny=clip_bbox_utm[1], 
            maxx=clip_bbox_utm[2], maxy=clip_bbox_utm[3],
            crs=target_crs
        )
        b04_clip = b04_clip.isel(y=slice(0, final_pixel_side), x=slice(0, final_pixel_side))
        clipped_data["B04"] = b04_clip.squeeze() 
        
        for asset_key, gsd in band_gsd_map.items():
            if asset_key == "B04": continue
            
            asset_href = pc.sign(item.assets[asset_key].href)
            ds = rioxarray.open_rasterio(asset_href)
            
            ds_clip = ds.rio.clip_box(
                minx=clip_bbox_utm[0], miny=clip_bbox_utm[1], 
                maxx=clip_bbox_utm[2], maxy=clip_bbox_utm[3],
                crs=target_crs
            )

            resampling_method = rasterio.enums.Resampling.nearest if asset_key == "SCL" else rasterio.enums.Resampling.bilinear
            ds_aligned = ds_clip.rio.reproject_match(b04_clip, resampling=resampling_method)
            
            if asset_key != "visual":
                clipped_data[asset_key] = ds_aligned.squeeze()
            else:
                 clipped_data[asset_key] = ds_aligned
                 
    except NoDataInBounds:
        print(f"Error: The calculated bounding box is outside the bounds of the satellite tile.")
        print(f"Tile Bounds: {b04_ds.rio.bounds()}")
        print(f"Requested Box: {clip_bbox_utm}")
        return False
    except Exception as e:
        print(f"An unexpected error occurred during clipping: {e}")
        return False
    
    # --- CALCULATE CLOUD & WATER STATS FROM SCL ---
    scl_vals = clipped_data["SCL"].values
    cloud_mask = (scl_vals == 3) | ((scl_vals >= 7) & (scl_vals <= 10))
    water_mask = (scl_vals == 6)
    
    calculated_per_clouds = round((np.sum(cloud_mask) / scl_vals.size) * 100, 2)
    calculated_water_pixels = int(np.sum(water_mask))
    
    print(f"Calculated Stats - Clouds: {calculated_per_clouds}%, Water Pixels: {calculated_water_pixels}")

    if save_data:
        output_dir = Path(output_path)
        output_dir.mkdir(parents=True, exist_ok=True)
        print("\n--- Saving RAW TIFFs ---")
        for key, da in clipped_data.items():
            da.rio.to_raster(output_dir / f"{key}_raw.tif")

    # 6. Calculate Indices
    SCALE_FACTOR = 10000.0 
    b04_ref = (clipped_data["B04"] / SCALE_FACTOR).astype(np.float32)
    b05_ref = (clipped_data["B05"] / SCALE_FACTOR).astype(np.float32)
    b07_ref = (clipped_data["B07"] / SCALE_FACTOR).astype(np.float32)
    b08_ref = (clipped_data["B08"] / SCALE_FACTOR).astype(np.float32)
    
    sum_b8_b4 = b08_ref + b04_ref
    ndvi = (b08_ref - b04_ref) / sum_b8_b4.where(sum_b8_b4 != 0, np.nan) 
    sum_b7_b5 = b07_ref + b05_ref
    ndci = (b07_ref - b05_ref) / sum_b7_b5.where(sum_b7_b5 != 0, np.nan) 
    
    vis_data = clipped_data.copy()
    vis_data["NDVI"] = ndvi
    vis_data["NDCI"] = ndci
    
    # 8. Save Visual Previews
    if save_data:
        print("--- Saving Visual Previews ---")
        
        colors = ['white', 'white', 'black', 'black', 'green', 'saddlebrown', 'lightblue', 'grey', 'grey', 'grey', 'grey']
        cmap_scl = ListedColormap(colors)
        bounds = np.arange(12)
        norm_scl = BoundaryNorm(bounds, cmap_scl.N)

        def save_plot_with_points(da, key, cmap=None, norm=None, vmin=None, vmax=None, rgb=False):
            fig, ax = plt.subplots(figsize=(10, 10))
            extent_utm = [da.x.min(), da.x.max(), da.y.min(), da.y.max()]
            
            if rgb:
                arr = da.transpose('y', 'x', 'band').values
                vmin, vmax = np.nanpercentile(arr, [2, 98])
                arr_scaled = np.clip((arr - vmin) / (vmax - vmin), 0, 1)
                ax.imshow(arr_scaled, extent=extent_utm, origin='upper')
            elif key == "SCL":
                ax.imshow(da.values, cmap=cmap, norm=norm, extent=extent_utm, origin='upper')
            else:
                robust = True if vmin is None else False
                arr_2d = da.values.squeeze()
                if robust:
                   vmin, vmax = np.nanpercentile(arr_2d, [2, 98])
                ax.imshow(arr_2d, cmap=cmap, extent=extent_utm, origin='upper', vmin=vmin, vmax=vmax)
            
            ax.set_axis_off() 
            plt.savefig(output_dir / f"{key}_preview.png", bbox_inches='tight', pad_inches=0)
            plt.close(fig)

        for key in vis_data.keys():
            da = vis_data[key]
            if key == "visual": save_plot_with_points(da, key, rgb=True)
            elif key == "SCL": save_plot_with_points(da, key, cmap=cmap_scl, norm=norm_scl)
            elif key == "NDVI": save_plot_with_points(da, key, cmap="RdYlGn", vmin=-1, vmax=1)
            elif key == "NDCI": save_plot_with_points(da, key, cmap="jet", vmin=-1, vmax=1)
            else: save_plot_with_points(da, key, cmap="gray")

        metadata = {
            "item_id": item.id,
            "per_clouds": calculated_per_clouds,
            "water_pixels": calculated_water_pixels,
            "uid": case_id, # Modified to use argument
            "abun": "N/A", # Not available in simple mode
            "tile_size_10m_pixels": final_pixel_side,
            "date": item.properties["datetime"].split('T')[0],
            "center_lat": center_lat,
            "center_lon": center_lon,
            "points_data": points_data_list
        }
        with open(output_dir / "metadata.json", "w") as f:
            json.dump(metadata, f, indent=4)
            
        print(f"Data successfully saved to: {output_dir.resolve()}")
        
    return True # Return Success

def predict_using_cyfi_pipeline(input_folder_path, 
                                date_str, 
                                metadata_filename="metadata.json",
                                print_images=False):
    
    input_dir = Path(input_folder_path).resolve()
    print(f"--- Starting CyFi Pipeline Integration (Cloud Threshold 7.5%): {input_dir} ---")

    # 1. Configuration
    features_config = FeaturesConfig()
    CLOUD_THRESHOLD = 0.075 
    features_config.max_cloud_percent = CLOUD_THRESHOLD

    WINDOW_METERS = features_config.image_feature_meter_window 
    PIXEL_SIZE = 10
    WINDOW_PIXELS = WINDOW_METERS // PIXEL_SIZE 
    RADIUS_PIXELS = WINDOW_PIXELS // 2 

    # 2. Load Raw Raster Data & Metadata
    required_bands = features_config.use_sentinel_bands 
    data_store = {}
    
    try:
        scl_da = rioxarray.open_rasterio(input_dir / "SCL_raw.tif").squeeze()
        data_store["SCL"] = scl_da.values
        height, width = scl_da.shape
        transform = scl_da.rio.transform()
        crs = scl_da.rio.crs
    except FileNotFoundError:
        print("Error: SCL_raw.tif not found. Run Part 1 first.")
        return (False, "")

    # Load Metadata (just for stats update later; reference points loading REMOVED)
    metadata_path = input_dir / metadata_filename
    metadata = {}
    
    try:
        with open(metadata_path, "r") as f:
            metadata = json.load(f)
            
    except FileNotFoundError:
        print(f"Warning: {metadata_filename} not found.")

    # Load other bands
    for band in required_bands:
        if band == "SCL": continue
        path = input_dir / f"{band}_raw.tif"
        if path.exists():
            data_store[band] = rioxarray.open_rasterio(path).squeeze().values
        else:
            data_store[band] = np.full((height, width), np.nan, dtype=np.float32)

    # 3. Generate Lattice Grid
    print("Generating 100m Lattice Grid...")
    GRID_STEP = 10 
    
    rows = np.arange(0, height, GRID_STEP)
    cols = np.arange(0, width, GRID_STEP)
    grid_rows, grid_cols = np.meshgrid(rows, cols, indexing='ij')
    grid_rows = grid_rows.flatten()
    grid_cols = grid_cols.flatten()
    
    valid_mask = (grid_rows < height) & (grid_cols < width)
    grid_rows = grid_rows[valid_mask]
    grid_cols = grid_cols[valid_mask]
    
    # Check water mask
    water_mask = (data_store["SCL"][grid_rows, grid_cols] == 6)
    target_rows = grid_rows[water_mask]
    target_cols = grid_cols[water_mask]
    
    initial_samples = len(target_rows)
    print(f"Identified {initial_samples} potential water points.")
    if initial_samples == 0: return (False, "")

    # 4. Create Mock Cache Structure
    temp_cache = Path(tempfile.mkdtemp(prefix="cyfi_lattice_"))
    
    cache_subdir = temp_cache / f"sentinel_{WINDOW_METERS}"
    fake_item_id = "LATTICE_ITEM" 

    valid_sample_ids = []
    valid_lats = []
    valid_lons = []
    valid_rows = []
    valid_cols = []

    transformer = Transformer.from_crs(crs, "EPSG:4326", always_xy=True)

    # 5. Filter & Slice Data
    print("Checking Cloud Cover & Slicing Data...")
    
    skipped_clouds = 0
    
    for idx, (r, c) in enumerate(tqdm(zip(target_rows, target_cols), total=initial_samples)):
        s_id = f"sample_{idx}"
        
        r_min = max(0, r - RADIUS_PIXELS)
        r_max = min(height, r + RADIUS_PIXELS)
        c_min = max(0, c - RADIUS_PIXELS)
        c_max = min(width, c + RADIUS_PIXELS)
        
        if (r_max - r_min) == 0 or (c_max - c_min) == 0:
            continue

        scl_window = data_store["SCL"][r_min:r_max, c_min:c_max]
        cloud_pixel_count = ((scl_window >= 7) & (scl_window <= 10)).sum()
        total_pixels = scl_window.size
        cloud_ratio = cloud_pixel_count / total_pixels
        
        if cloud_ratio > CLOUD_THRESHOLD:
            skipped_clouds += 1
            continue

        item_dir = cache_subdir / s_id / fake_item_id
        item_dir.mkdir(parents=True, exist_ok=True)
        
        x, y = rasterio.transform.xy(transform, r, c)
        lon, lat = transformer.transform(x, y)
        
        valid_sample_ids.append(s_id)
        valid_lats.append(lat)
        valid_lons.append(lon)
        valid_rows.append(r)
        valid_cols.append(c)

        for band in required_bands:
            arr_window = data_store[band][r_min:r_max, c_min:c_max]
            arr_reshaped = arr_window[np.newaxis, :, :] 
            np.save(item_dir / f"{band}.npy", arr_reshaped)

    print("\nProcessing Summary:")
    print(f" - Total Potential Points: {initial_samples}")
    print(f" - Skipped (Cloud > {CLOUD_THRESHOLD*100}%): {skipped_clouds}")
    print(f" - Valid Points to Predict: {len(valid_sample_ids)}")

    if len(valid_sample_ids) == 0:
        print("No valid points remained after cloud filtering.")
        shutil.rmtree(temp_cache)
        return (False, "")

    # 6. Prepare DataFrames for CyFi
    samples_df = pd.DataFrame({
        "sample_id": valid_sample_ids,
        "date": [date_str] * len(valid_sample_ids),
        "latitude": valid_lats,
        "longitude": valid_lons
    }).set_index("sample_id")

    satellite_meta_df = pd.DataFrame({
        "sample_id": valid_sample_ids,
        "item_id": [fake_item_id] * len(valid_sample_ids),
        "days_before_sample": [0] * len(valid_sample_ids), 
        "datetime": [date_str] * len(valid_sample_ids), 
        "visual_href": [None] * len(valid_sample_ids) 
    })

    # 7. Run CyFi Feature Generation
    print("Running CyFi Feature Generation...")
    try:
        _, features_df = generate_all_features(
            samples=samples_df,
            satellite_meta=satellite_meta_df,
            config=features_config,
            cache_dir=temp_cache
        )
    
    except SystemExit as e:
        print(f"   > CyFi triggered SystemExit (likely no valid data): {e}")
        shutil.rmtree(temp_cache)
        return (False, "")
    except Exception as e:
        print(f"Feature generation failed: {e}")
        shutil.rmtree(temp_cache)
        return (False, "")

    # 8. Run CyFi Prediction
    print("Running Prediction...")
    pipeline = CyFiPipeline.from_disk(DEFAULT_MODEL_PATH)
    pipeline.predict_features = features_df
    pipeline.predict_samples = samples_df
    pipeline._predict_model()
    
    results_df = pipeline.output_df.reset_index()
    results_df["pixel_row"] = valid_rows
    results_df["pixel_col"] = valid_cols
    
    csv_path = input_dir / "cyfi_lattice_predictions.csv"
    results_df.to_csv(csv_path, index=False)
    print(f"Predictions saved to {csv_path}")

    # --- UPDATE METADATA JSON ---
    severity_list = results_df.severity.astype(str).str.lower().to_list()
    
    count_high = severity_list.count("high")
    count_moderate = severity_list.count("moderate")
    count_low = severity_list.count("low")

    metadata["High counts"] = count_high
    metadata["Moderate counts "] = count_moderate
    metadata["Low counts "] = count_low
    
    with open(metadata_path, "w") as f:
        json.dump(metadata, f, indent=4)
    print(f"Updated metadata saved to: {metadata_path}")

    # 9. Visualization (MODIFIED: NO REFERENCE POINTS)
    # ---------------------------------------------------------
    print("Generating Visualization...")
    try:
        raw_vis_path = input_dir / "visual_raw.tif"
        if raw_vis_path.exists():
            raw_vis = rioxarray.open_rasterio(raw_vis_path).squeeze()
            bg_img = np.moveaxis(raw_vis.values, 0, -1)
            
            vmin, vmax = np.nanpercentile(bg_img, [2, 98])
            bg_img = np.clip((bg_img - vmin) / (vmax - vmin), 0, 1)

            fig, ax = plt.subplots(figsize=(12, 12))
            ax.imshow(bg_img)
            
            # 1. PREDICTIONS (Dots)
            severity_colors_pred = {
                'low': 'green', 'moderate': 'orange', 'high': 'red',
                '1': 'green', '2': 'orange', '3': 'red',
                1: 'green', 2: 'orange', 3: 'red'
            }
            results_df['severity'] = results_df['severity'].astype(str)
            colors_pred = results_df['severity'].map(lambda x: severity_colors_pred.get(x.lower(), 'gray'))
            
            ax.scatter(results_df['pixel_col'], results_df['pixel_row'], 
                       c=colors_pred, s=20, alpha=0.9, edgecolors='black', linewidth=0.5, label='Prediction')
            
            # 2. REFERENCE POINTS (REMOVED)
            # and plotted 'X' markers is deleted.

            legend_elements = [
                Line2D([0], [0], marker='o', color='w', markerfacecolor='green', label='Low', markersize=8),
                Line2D([0], [0], marker='o', color='w', markerfacecolor='orange', label='Moderate', markersize=8),
                Line2D([0], [0], marker='o', color='w', markerfacecolor='red', label='High', markersize=8)
            ]
            ax.legend(handles=legend_elements, loc='upper right')
            
            ax.set_title(f"CyFi Predictions: {date_str} (Cloud < {CLOUD_THRESHOLD*100}%)")
            ax.set_axis_off()
            
            final_img_path = input_dir / "cyfi_prediction_map.png"
            plt.savefig(final_img_path, bbox_inches='tight', dpi=150)
            print(f"Visualization SAVED to: {final_img_path}")
            
            if print_images:
                plt.show()
            plt.close(fig)
            
    except Exception as e:
        print(f"Visualization failed: {e}")
        import traceback
        traceback.print_exc()

    print("Cleaning up temporary cache...")
    shutil.rmtree(temp_cache)
    
    time.sleep(1)
    
    # 10. Rename Folder
    try:
        new_folder_name = f"{input_dir.name}_H{count_high}_M{count_moderate}_L{count_low}"
        if not input_dir.name.endswith(f"_H{count_high}_M{count_moderate}_L{count_low}"):
            new_folder_path = input_dir.parent / new_folder_name
            input_dir.rename(new_folder_path)
            print(f"Successfully renamed folder to: {new_folder_path}")
            return (True, new_folder_path)
        return (True, input_dir)
        
    except Exception as e:
        print(f"Could not rename folder: {e}")
        # Even if rename fails, return True for folder_ok but empty string for new path
        return (True, str(input_dir))
    

def update_uids_and_summary(folder_path, uids_file, summary_file):
    
    folder = Path(folder_path)
    
    # 1. Load Metadata
    try:
        with open(folder / "metadata.json") as f:
            meta = json.load(f)
    except FileNotFoundError:
        print(f"Error: metadata.json not found in {folder}")
        return

    # --- 2. UID TRACKER (Modified for CSV & Top-level UID) ---
    current_uid = meta.get("uid", "N/A")

    # UID tracker
    if Path(uids_file).exists():
        df_u = pd.read_excel(uids_file)
    else:
        df_u = pd.DataFrame(columns=["uid"])

    # Add new UID and save
    new_row = pd.DataFrame({"uid": [current_uid]})
    df_u = pd.concat([df_u, new_row], ignore_index=True).drop_duplicates(subset=['uid'])
    df_u.to_excel(uids_file, index=False)

    # --- 3. SUMMARY (Excel) ---
    # Use meta.get for 'case' instead of parsing filename (safer)
    # Note: 'points_data' inside meta is now a list with one dict, 
    # but we can just grab top-level data or the first item.
    
    points_data_list = meta.get("points_data", [])
    first_point = points_data_list[0] if points_data_list else {}

    row = {
        "uid": current_uid,
        "case": meta.get("case", first_point.get("case", "N/A")), 
        "item_id": meta.get("item_id", "N/A"),
        "date": meta.get("date", "N/A"),
        "lat": meta.get("center_lat", "N/A"),
        "lon": meta.get("center_lon", "N/A"),
        "abun": meta.get("abun", "N/A"),
        "abun_list": "N/A", # No list in this workflow
        "per_clouds": meta.get("per_clouds", "N/A"), 
        "water_pixels": meta.get("water_pixels", "N/A"),
        "high_pred": meta.get("High counts", 0),
        "mod_pred": meta.get("Moderate counts ", 0),
        "low_pred": meta.get("Low counts ", 0),
        "pred_visual": "", 
        "source_path": str(folder)
    }

    if Path(summary_file).exists():
        df_s = pd.read_excel(summary_file)
    else:
        df_s = pd.DataFrame()

    df_s = pd.concat([df_s, pd.DataFrame([row])], ignore_index=True)
    df_s.to_excel(summary_file, index=False)
    
    print(f"Summary updated for {current_uid}")

In [45]:
# Dictionary mapping your 4 scenarios to their saved weight files
MODEL_PATHS = {
    "res18_scl": r"C:\Users\KostasPikounis\OneDrive_Inlecom_Personal\OneDrive - INLECOM\Amfitrite\task2\IWD CNN\res18_2classes\results11\best_epoch_16.pth",
    "res18_no_scl": r"C:\Users\KostasPikounis\OneDrive_Inlecom_Personal\OneDrive - INLECOM\Amfitrite\task2\IWD CNN\res18_2classes\results12\best_epoch_26.pth",
    "convnext_scl": r"C:\Users\KostasPikounis\OneDrive_Inlecom_Personal\OneDrive - INLECOM\Amfitrite\task2\IWD CNN\convnextv2_base\results3_new\best_epoch_15.pth",
    "rdnet_no_scl": r"C:\Users\KostasPikounis\OneDrive_Inlecom_Personal\OneDrive - INLECOM\Amfitrite\task2\IWD CNN\rdnet_base\results3\best_epoch_35.pth"
}

In [126]:
def load_tensor_from_folder(folder_path, use_mask=False, img_size=256):
    """Loads bands from a single folder, applies mask if needed, and returns a model-ready tensor."""
    band_names = [
        "B02_raw.tif", "B03_raw.tif", "B04_raw.tif", "B05_raw.tif", "B06_raw.tif",
        "B07_raw.tif", "B08_raw.tif", "B8A_raw.tif", "B11_raw.tif", "B12_raw.tif"
    ]
    
    band_data = []
    for b_name in band_names:
        p = os.path.join(folder_path, b_name)
        if not os.path.exists(p):
            raise FileNotFoundError(f"Missing required band file: {p}")
            
        with rasterio.open(p) as src:
            band_data.append(src.read(1).astype(np.float32))
            
    bands_stack = np.stack(band_data, axis=0)
    
    # Apply Mask (If requested)
    if use_mask:
        scl_path = os.path.join(folder_path, "SCL_raw.tif")
        if os.path.exists(scl_path):
            with rasterio.open(scl_path) as src:
                scl = src.read(1)
            # SCL 6 is water
            mask = (scl == 6).astype(np.float32)
            bands_stack = bands_stack * mask
        else:
            print("  [Warning] SCL_raw.tif not found in folder. Proceeding without mask.")
            
    # Convert to Tensor and add Batch Dimension -> Shape: (1, 10, H, W)
    tensor = torch.from_numpy(bands_stack).unsqueeze(0)
    
    # Resize to expected dimensions
    tensor = torch.nn.functional.interpolate(
        tensor, size=(img_size, img_size), 
        mode='bilinear', align_corners=False
    )
    
    # Normalize
    tensor = tensor / 10000.0
    
    return tensor


class HABLightningModel(L.LightningModule):
    def __init__(self, arch_name='resnet18', num_classes=2, in_chans=10):
        super().__init__()
        self.model = self._build_model(arch_name, num_classes, in_chans)

    def _build_model(self, arch, num_classes, in_chans):
        if arch == 'resnet18':
            model = models.resnet18(weights=None)
            model.conv1 = nn.Conv2d(in_chans, 64, kernel_size=7, stride=2, padding=3, bias=False)
            model.fc = nn.Linear(model.fc.in_features, num_classes)
            return model
        elif arch == 'convnextv2_base':
            return timm.create_model('convnextv2_base', pretrained=False, num_classes=num_classes, in_chans=in_chans)
        elif arch == 'rdnet_base':
            return timm.create_model('rdnet_base', pretrained=False, num_classes=num_classes, in_chans=in_chans)
        else:
            raise ValueError(f"Unknown architecture: {arch}")

    def forward(self, x):
        return self.model(x)

def run_single_folder(target_folder):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Running on device: {device}")
    
    # Define output text file path
    output_txt_path = os.path.join(target_folder, "cnn_results.txt")
    
    # Open the text file for writing
    with open(output_txt_path, "w") as f:
        header = f"Analyzing Folder: {target_folder}\n" + "-" * 50
        print(header)
        f.write(header + "\n")
        
        # Scenarios: (Key, Architecture, Use Mask?, Image Size)
        scenarios = [
            ("res18_scl",      "resnet18",        True,  256),
            ("res18_no_scl",   "resnet18",        False, 256),
            ("convnext_scl",   "convnextv2_base", True,  256),
            ("rdnet_no_scl",   "rdnet_base",      False, 256)
        ]
        
        for scenario_name, arch, use_mask, img_size in scenarios:
            # 1. Load Data
            input_tensor = load_tensor_from_folder(target_folder, use_mask=use_mask, img_size=img_size)
            input_tensor = input_tensor.to(device)
            
            # 2. Setup Model & Load Weights
            model_wrapper = HABLightningModel(arch_name=arch)
            ckpt_path = MODEL_PATHS.get(scenario_name)
            
            if not ckpt_path or not os.path.exists(ckpt_path):
                error_msg = f"{scenario_name:<15} | ERROR: Weights not found at {ckpt_path}"
                print(error_msg)
                f.write(error_msg + "\n")
                continue
                
            # Handle state_dict loading
            if ckpt_path.endswith('.ckpt'):
                checkpoint = torch.load(ckpt_path, map_location='cpu')
                state_dict = {k.replace('model.', ''): v for k, v in checkpoint['state_dict'].items()}
                model_wrapper.model.load_state_dict(state_dict, strict=False)
            else:
                state_dict = torch.load(ckpt_path, map_location='cpu')
                model_wrapper.model.load_state_dict(state_dict, strict=False)
                
            model_wrapper.to(device)
            model_wrapper.eval()
            
            # 3. Predict
            with torch.no_grad():
                outputs = model_wrapper(input_tensor)
                _, preds = torch.max(outputs, 1)
                pred_class = preds.item()
                
            # 4. Map and Print Result
            str_pred = "YES" if pred_class == 1 else "NO"
            result_line = f"Model: {scenario_name:<15} | HAB Detected: {str_pred}"
            
            # Write to console and file
            print(result_line)
            f.write(result_line + "\n")

        footer = "-" * 50
        print(footer)
        f.write(footer + "\n")
        
    print(f"\nResults have been successfully saved to: {output_txt_path}")

In [112]:
SEARCH_BUFFER_METERS = 3000
CLOUD_CUTOFF = 7.5
OUTPUT_ROOT_FOLDER = r"C:\Users\KostasPikounis\OneDrive_Inlecom_Personal\OneDrive - INLECOM\Amfitrite\task2\OWD\test_data1"

In [113]:
'''
case_id = "HAEDAT:COL-02:CO-16-002"
lat = 10.840556
lon = -74.523056
dates = ["05-02-2017"]

###################################

case_id = "HAEDAT:COL-02:CO-16-003"
lat = 10.832778	
lon = -74.573889
dates = ["21-12-2016"]

###################################

case_id = "HAEDAT:COL-02:CO-16-004"
lat = 10.990556	
lon = -74.281667
dates = ["10-12-2016"]

###################################

case_id = "HAEDAT:GR-01:GR-19-001"
lat = 40.549722
lon = 22.8
dates = ["27-11-2019"]
date_str = dates[0]
uid = f"{case_id}_{date_str}"
'''

'\ncase_id = "HAEDAT:COL-02:CO-16-002"\nlat = 10.840556\nlon = -74.523056\ndates = ["05-02-2017"]\n\n###################################\n\ncase_id = "HAEDAT:COL-02:CO-16-003"\nlat = 10.832778\t\nlon = -74.573889\ndates = ["21-12-2016"]\n\n###################################\n\ncase_id = "HAEDAT:COL-02:CO-16-004"\nlat = 10.990556\t\nlon = -74.281667\ndates = ["10-12-2016"]\n\n###################################\n'

In [140]:
BOX_SIDE_PIXELS = 256  # 365 pixels * 10m = 3650m box
case_id = "HAEDAT:GR-01:GR-19-003_500m_to_the_left"
lat = 40.638889
lon = 22.887
dates = ["18-01-2019"]
date_str = dates[0]
uid = f"{case_id}_{date_str}"

In [141]:
items_df = find_satelite_images(lat, lon, date_str, meter_buffer=SEARCH_BUFFER_METERS)
items_df

Search BBox: [22.851535947054664, 40.611873332687445, 22.922464052945337, 40.665904540520664]


C:\Users\KostasPikounis\AppData\Local\Temp\ipykernel_20616\2360138925.py:21: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  range_start = pd.to_datetime(date) - timedelta(days=time_buffer_days)
C:\Users\KostasPikounis\AppData\Local\Temp\ipykernel_20616\2360138925.py:22: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  range_end = pd.to_datetime(date) + timedelta(days=time_buffer_days)


Filtering from 12 returned to 12 items that contain the sample location


C:\Users\KostasPikounis\AppData\Local\Temp\ipykernel_20616\2360138925.py:141: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  item_details['date_difference'] = (pd.to_datetime(item_details['datetime']) - pd.to_datetime(sel_date)).dt.days


,datetime,platform,min_long,max_long,min_lat,max_lat,bbox,item_obj,contains_sample_point,per_clouds,date_difference
0,2019-01-29,Sentinel-2A,22.181148,23.514538,40.536186,41.545594,"[22.181147928456813, 40.536185528423594, 23.51...",<Item id=S2A_MSIL2A_20190129T092241_R093_T34TF...,True,100.00,11
1,2019-01-29,Sentinel-2A,22.165666,23.480449,39.635871,40.644800,"[22.1656655200817, 39.63587147039632, 23.48044...",<Item id=S2A_MSIL2A_20190129T092241_R093_T34TF...,True,100.00,11
2,2019-01-24,Sentinel-2B,22.181148,23.514538,40.536186,41.545594,"[22.181147928456813, 40.536185528423594, 23.51...",<Item id=S2B_MSIL2A_20190124T092259_R093_T34TF...,True,3.75,6
3,2019-01-24,Sentinel-2B,22.165666,23.480449,39.635871,40.644800,"[22.1656655200817, 39.63587147039632, 23.48044...",<Item id=S2B_MSIL2A_20190124T092259_R093_T34TF...,True,27.75,6
4,2019-01-19,Sentinel-2A,22.181148,23.514538,40.536186,41.545594,"[22.181147928456813, 40.536185528423594, 23.51...",<Item id=S2A_MSIL2A_20190119T092311_R093_T34TF...,True,100.00,1
5,2019-01-19,Sentinel-2A,22.165666,23.480449,39.635871,40.644800,"[22.1656655200817, 39.63587147039632, 23.48044...",<Item id=S2A_MSIL2A_20190119T092311_R093_T34TF...,True,100.00,1
6,2019-01-14,Sentinel-2B,22.181148,23.514538,40.536186,41.545594,"[22.181147928456813, 40.536185528423594, 23.51...",<Item id=S2B_MSIL2A_20190114T092339_R093_T34TF...,True,100.00,-4
7,2019-01-14,Sentinel-2B,22.165666,23.480449,39.635871,40.644800,"[22.1656655200817, 39.63587147039632, 23.48044...",<Item id=S2B_MSIL2A_20190114T092339_R093_T34TF...,True,100.00,-4
8,2019-01-09,Sentinel-2A,22.181148,23.514538,40.536186,41.545594,"[22.181147928456813, 40.536185528423594, 23.51...",<Item id=S2A_MSIL2A_20190109T092351_R093_T34TF...,True,100.00,-9
9,2019-01-09,Sentinel-2A,22.165666,23.480449,39.635871,40.644800,"[22.1656655200817, 39.63587147039632, 23.48044...",<Item id=S2A_MSIL2A_20190109T092351_R093_T34TF...,True,100.00,-9


In [142]:
best_item_index = 2
selected_item = items_df.iloc[best_item_index]["item_obj"]
final_date = selected_item.datetime.strftime("%Y-%m-%d")
out_folder_name = f"case{case_id}_{final_date}_item{best_item_index}".replace(":", "")
out_folder_path = os.path.join(OUTPUT_ROOT_FOLDER, out_folder_name)
if not os.path.exists(out_folder_path):
    os.makedirs(out_folder_path)


In [143]:
extract_ok = extract_and_save_tile(
    item=selected_item,
    lat=lat,
    lon=lon,
    case_id=uid, # Passing case_id for metadata
    initial_pixel_size=BOX_SIDE_PIXELS,
    save_data=True,
    output_path=out_folder_path,
    print_images=True
)

--- Starting Tile Extraction and Processing (Single Center) ---
Center Point (Lat/Lon): (40.6389, 22.8870)
Final Tile Dimension (10m pixels): 256 x 256
--- Clipping and Aligning Rasters ---
Calculated Stats - Clouds: 5.1%, Water Pixels: 36844

--- Saving RAW TIFFs ---
--- Saving Visual Previews ---
Data successfully saved to: C:\Users\KostasPikounis\OneDrive_Inlecom_Personal\OneDrive - INLECOM\Amfitrite\task2\OWD\test_data1\caseHAEDATGR-01GR-19-003_500m_to_the_left_2019-01-24_item2


In [144]:
 pred_ok, final_path = predict_using_cyfi_pipeline(out_folder_path, final_date)

--- Starting CyFi Pipeline Integration (Cloud Threshold 7.5%): C:\Users\KostasPikounis\OneDrive_Inlecom_Personal\OneDrive - INLECOM\Amfitrite\task2\OWD\test_data1\caseHAEDATGR-01GR-19-003_500m_to_the_left_2019-01-24_item2 ---
Generating 100m Lattice Grid...
Identified 372 potential water points.
Checking Cloud Cover & Slicing Data...


100%|██████████| 372/372 [00:07<00:00, 50.56it/s]
2026-02-20 13:38:04.323 | INFO     | cyfi.data.features:calculate_satellite_features:48 - Generating satellite features for 371 images.



Processing Summary:
 - Total Potential Points: 372
 - Skipped (Cloud > 7.5%): 1
 - Valid Points to Predict: 371
Running CyFi Feature Generation...


100%|██████████| 371/371 [00:08<00:00, 42.37it/s] 
2026-02-20 13:38:14.207 | INFO     | cyfi.data.features:calculate_satellite_features:64 - Dropping 0 row(s) where bounding box has too many clouds.
2026-02-20 13:38:14.208 | INFO     | cyfi.data.features:calculate_satellite_features:73 - Dropping 0 row(s) where bouding box does not contain any water.
2026-02-20 13:38:14.210 | INFO     | cyfi.data.features:calculate_satellite_features:79 - Dropping 0 row(s) where bouding box contains missing pixels.
2026-02-20 13:38:14.253 | INFO     | cyfi.data.features:calculate_metadata_features:219 - Generating land cover features for 371 sample points.
100%|██████████| 371/371 [00:07<00:00, 51.41it/s] 
2026-02-20 13:38:22.600 | INFO     | cyfi.data.features:generate_all_features:317 - Generated 29 satellite feature(s) and 1 sample metadata feature(s) for 371 sample points (100% of sample points)


Running Prediction...
Predictions saved to C:\Users\KostasPikounis\OneDrive_Inlecom_Personal\OneDrive - INLECOM\Amfitrite\task2\OWD\test_data1\caseHAEDATGR-01GR-19-003_500m_to_the_left_2019-01-24_item2\cyfi_lattice_predictions.csv
Updated metadata saved to: C:\Users\KostasPikounis\OneDrive_Inlecom_Personal\OneDrive - INLECOM\Amfitrite\task2\OWD\test_data1\caseHAEDATGR-01GR-19-003_500m_to_the_left_2019-01-24_item2\metadata.json
Generating Visualization...
Visualization SAVED to: C:\Users\KostasPikounis\OneDrive_Inlecom_Personal\OneDrive - INLECOM\Amfitrite\task2\OWD\test_data1\caseHAEDATGR-01GR-19-003_500m_to_the_left_2019-01-24_item2\cyfi_prediction_map.png
Cleaning up temporary cache...
Could not rename folder: [WinError 5] Access is denied: 'C:\\Users\\KostasPikounis\\OneDrive_Inlecom_Personal\\OneDrive - INLECOM\\Amfitrite\\task2\\OWD\\test_data1\\caseHAEDATGR-01GR-19-003_500m_to_the_left_2019-01-24_item2' -> 'C:\\Users\\KostasPikounis\\OneDrive_Inlecom_Personal\\OneDrive - INLECOM\

In [145]:
run_single_folder(final_path)

Running on device: cpu
Analyzing Folder: C:\Users\KostasPikounis\OneDrive_Inlecom_Personal\OneDrive - INLECOM\Amfitrite\task2\OWD\test_data1\caseHAEDATGR-01GR-19-003_500m_to_the_left_2019-01-24_item2
--------------------------------------------------
Model: res18_scl       | HAB Detected: YES
Model: res18_no_scl    | HAB Detected: YES
Model: convnext_scl    | HAB Detected: YES
Model: rdnet_no_scl    | HAB Detected: YES
--------------------------------------------------

Results have been successfully saved to: C:\Users\KostasPikounis\OneDrive_Inlecom_Personal\OneDrive - INLECOM\Amfitrite\task2\OWD\test_data1\caseHAEDATGR-01GR-19-003_500m_to_the_left_2019-01-24_item2\cnn_results.txt
